<a href="https://colab.research.google.com/github/vikramvundyala/python_AI-ML/blob/main/Hackathon4b_Expression_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

Automated facial expression recognition provides an objective assessment of emotions. Human based assessment of emotions has many limitations and biases and automated facial expression technology has been found to deliver a better level of insight into behavior patterns. Emotion detection from facial expressions using AI is useful in automatically measuring consumers’ engagement with their content and brands, audience engagement for advertisements, customer satisfaction in the retail sector, psychological analyses, law enforcement etc.

**Objectives:**

**Stage 4 (20 Marks):** Train a CNN Model, update your HuggingFace Space repository, and test your Deployment for Expression Recognition on the HuggingFace Space App.

##**Stage 4 (20 Marks)**

**(i) Train a CNN Model for Expression Recognition on given Expression data**

**(ii) Deploy the Model and Perform Expression Recognition on Team Data through the HuggingFace Space App**


---


* Define and train a CNN for expression recognition for the data under folder `"Expression_data"` which segregated on expression basis.

* Collect your team data by running the code cells provided within the notebook below.

* Test your model on the collected team data and optimize the CNN architecture for predicting the respective labels of the images.

- Update files present in your cloned HuggingFace Space repository.

    - Save and Download the trained expression model (`expression_model.t7`) and upload/place it in your HuggingFace Space repository within **`app/Hackathon_setup/`** folder.
    
    - Update the model architecture in the **`app/Hackathon_setup/exp_recognition_model.py`** file.
    
    - Update the code in the **`get_expression()`** function of the **`app/Hackathon_setup/exp_recognition.py`** file. (See Deployment related files)

- Commit your changes and push to HuggingFace Space repository.

- Access the `App` tab of your repository to see the build progress (debug if error persists) [This step might take sometime.]

- Once the app has built successfully, you should see below message

    `Application startup complete. Uvicorn running on http://0.0.0.0:8001`

- Test the model's Expression Recognition functionality using the application running on your Space

    Go to 3-dots icon beside Settings, then select `Embed this Space` option, and go to `Direct URL`
    - Select the task, `Expression Recognition`
    - Select 'Send Anyway' when prompted
    - Upload your image and test
    - **NOTE:** When using the Direct URL link via android mobile, the camera option will also enable to capture images (Set 1:1 aspect ratio in camera settings before-hand)


### **Download the dataset**

In [ ]:
#@title Run this cell to download the dataset

from IPython import get_ipython
ipython = get_ipython()

notebook="M3_Hackathon" #name of the notebook

def setup():
#  ipython.magic("sx pip3 install torch")
    ipython.magic("sx wget wget https://cdn.talentsprint.com/aiml/Experiment_related_data/Expression_data.zip")
    ipython.magic("sx unzip Expression_data.zip")
    ipython.magic("sx wget https://cdn.iisc.talentsprint.com/AIandMLOps/Datasets/lbpcascade_frontalface.xml")

    print ("Setup completed successfully")
    return
setup()

In [ ]:
%ls

**Dataset attributes:**

During the setup you have downloaded the `Expression_data`:

* **Expression_data**: In this folder, the images are segregrated in terms of Expression
> * Expressions available: ANGER, DISGUST, FEAR, HAPPINESS, NEUTRAL, SADNESS, SURPRISE
> * Each class is organised as one folder
> * There are ~18000 total images in the training data and ~4500 total images in the testing data

### **Import Required Packages**

In [ ]:
%matplotlib inline
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torch.utils.data import DataLoader,Dataset
import matplotlib.pyplot as plt
import torchvision.utils
import numpy as np
import random
from PIL import Image                   # PIL (Pillow) is the Python Image Library. Used to cut and resize images, or do simple manipulation.
import torch
from torch.autograd import Variable
import PIL.ImageOps
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import os
import warnings
from time import sleep
import sys
warnings.filterwarnings('ignore')

For the following step, to obtain hints on building a CNN model for face expression, you may refer to this [article](https://drive.google.com/open?id=1P2rpaWW3tOtGGnw4dvtdZ4hjoc8iDNst).

**Define and train a CNN model for expression recognition**

In [ ]:
#YOUR CODE HERE : Sample Helper function


In [ ]:
#YOUR CODE HERE : Check number of training and Validation images


In [ ]:
#YOUR CODE HERE : Generate a batch of 10 images and labels


In [ ]:
# YOUR CODE HERE : Print the summary of the model

**Test your model and optimize CNN architecture for predicting the labels correctly**

In [ ]:
# YOUR CODE HERE for test evaluation

#### **Team Data Collection**

**Collect your team data and fine-tune the CNN for expression data on your team**

- Collect Team Data by running the code cells provided below
    - The collected Expression images of your team will be stored in the `captured_images_with_Expression` directory

    - NOTE: *Since team members will be using separate colab notebooks, they can capture their own face images and then download it and share with other members for model testing.* Code cell is provided below to download the data.

- This data will be useful for testing the above trained cnn network

In [ ]:
# @title Run this cell to Setup Image Capturing in Colab {display-mode: "form"}

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import imageio
import datetime
import pathlib
import cv2
import numpy as np
import matplotlib.pyplot as plt

AREA_THRESHOLD = 2304

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)

  im = Image.open(filename)
  im1 = im.crop((80, 0, 560, 480))
  im1.save(filename)

  return filename


def save_faces(miniframe, filepath):
    TRAINSET = "lbpcascade_frontalface.xml"
    classifier = cv2.CascadeClassifier(TRAINSET)
    faces = classifier.detectMultiScale(miniframe)
    image = get_large_face(miniframe, faces)
    if not isinstance(image, np.ndarray):
        return {"status" : False}
    plt.imshow(image)
    plt.show()
    #cv2.imwrite(filepath, image)
    imageio.imwrite(filepath, image)
    return {"status" : True}

def get_large_face(miniframe, faces):
    images = []
    face_areas = []
    required_image = 0
    for x,y,w,h in faces:
        face_cropped = miniframe[y:y+h, x:x+w]
        face_areas.append(w*h)
        images.append(face_cropped)
        required_image = images[np.argmax(face_areas)]
    if not face_areas:
        return 0
    if face_areas[np.argmax(face_areas)] < AREA_THRESHOLD:
        return 0

    return required_image

def save_image(filename, class_name):
    base_path = "captured_images_with_Expression/"

    pathlib.Path(base_path + class_name).mkdir(parents=True, exist_ok=True)
    file_name = class_name + "_" + datetime.datetime.now().strftime("%s") + ".jpg"
    filepath = base_path +class_name + "/" + file_name

    image = Image.open(filename)
    miniframe = np.asarray(image)
    status = save_faces(miniframe, filepath)
    if status['status']:
        print("Image saved in " + filepath, flush = True)
    else:
        print("Face not found!\nRetry!")


In [ ]:
# @title Capture an Image for ANGER $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "ANGER"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for DISGUST $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "DISGUST"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for FEAR $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "FEAR"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for HAPPINESS $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "HAPPINESS"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for NEUTRAL $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "NEUTRAL"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for SADNESS $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "SADNESS"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for SURPRISE $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "SURPRISE"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Download Collected Images $ $ [OPTIONAL]  {display-mode: "form"}

from google.colab import files
!zip -r "captured_images_with_Expression.zip" "captured_images_with_Expression"
files.download('captured_images_with_Expression.zip')
print("Downloaded captured_images_with_Expression.zip !!")

In [ ]:
%ls

**While uploading the team images manually to captured_images. this file .ipynb_checkpoints will be created and make issue to delet run the below code**

In [ ]:
rm -rf `find -type d -name .ipynb_checkpoints`

In [ ]:
# YOUR CODE HERE for loading the team expression data. Note: Use the same transform which used for Expression_Data.
# YOU CODE HERE for Dataloader

In [ ]:
# YOUR CODE HERE for getting the CNN representation of your team data with expression. Optimize the CNN model for predicting the labels of expressions correctly
# Note: If the CNN Model is not performing as expected, then you can add your Team Data to the Existing Training Data and Re-Train the Model.

**Save your trained model**

* Save the state dictionary of the classifier (use pytorch only), It will be useful in
integrating model to the mobile app

 [Hint](https://pytorch.org/tutorials/beginner/saving_loading_models.html)

In [ ]:
### YOUR CODE HERE for saving the CNN model

**Download your trained model**
* Given the path of model file the following code downloads it through the browser

In [ ]:
from google.colab import files
files.download('expression_model.t7')
#files.download('<model_file_path>')